# 77장 · 컨테이너 이미지 digest와 롤백 기준

태그는 다른 이미지를 가리키도록 옮길 수 있습니다. 배포 후보를 구별하려면 이미지 내용에 대응하는 digest와 기능 검사 결과를 함께 봅니다.

승인한 이미지와 배포 후보의 태그는 같지만 digest가 다른 사례를 다룹니다. 로컬 배포 기록으로 적용 조건과 실패 시 이전 이미지로 복구하는 기준을 점검합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


### 후보가 바뀌었는지 확인하는 두 값

<p>검토할 때 승인한 바이트열과 적용 직전의 후보 바이트열을 따로 보관합니다. 화면의 이름이 같아 보여도 계산한 지문이 다르면 서로 다른 내용입니다.</p>


<h3>적용 전과 적용 뒤의 서로 다른 중단 조건</h3>

<p>첫 셀에서는 승인 지문 B와 후보 지문 C가 달라 적용을 보류합니다. 뒤의 독립된 예제는 딕셔너리에서 A 대신 B를 활성화한 상황으로 시작합니다. B의 health가 200이어도 출처가 None이어서 포인터는 A로 돌아갑니다.</p>


## 가변 태그와 콘텐츠 지문

CI는 검사한 소스와 이미지 후보를 함께 인계합니다. CD는 환경 승인과 현재 배포 상태를 확인하고 그 후보만 적용합니다. 셀 01의 입력은 실제 이미지 파일이 아니라 course-image B와 course-image C라는 두 바이트열입니다. 이 두 값의 SHA-256부터 비교합니다. 실제 레지스트리에서 받은 manifest digest를 대신하는 수업용 값입니다.


### 같은 지문과 승인 여부를 함께 확인하기

<p>can_apply는 expected와 candidate가 같은지, 검토자가 승인했는지를 함께 봅니다. 지문이 같아도 승인하지 않은 후보는 적용하지 않습니다.</p>


### 승인한 이미지와 후보 이미지가 달라졌을 때

수업에서는 두 바이트열의 해시로 후보의 일치 여부를 연습합니다. 실제 레지스트리에서 내려받은 이미지를 검사한 결과와 구별하세요.

1. 승인본 B · 승인 대상 = course-image B
2. 후보 C · 현재 후보 = course-image C
3. 지문 비교 · approved == candidate = False
4. 적용 보류 · can_apply = False


### 셀 01 · 태그와 해시 지문 비교

내용 한 줄이 바뀌었을 때 같은 승인값을 계속 쓸 수 있는지 확인하세요.


In [1]:
import hashlib
def digest(data):
    return "sha256:" + hashlib.sha256(data).hexdigest()
approved = digest(b"course-image B")
candidate = digest(b"course-image C")
print(approved == candidate)

False


False는 B와 C의 지문이 다르다는 비교 결과입니다. 태그만 기록했다면 같은 이름이 다른 내용으로 옮겨간 뒤 어느 후보를 승인했는지 구분하기 어렵습니다.


### 과제 02 · 적용 전 조건 검사

사람이 승인한 지문과 현재 후보 지문을 비교해 적용 여부를 반환하세요. 지문 불일치, 승인 누락, 두 조건 충족을 각각 검사하세요.


In [2]:
def can_apply(expected, candidate, approved_by_reviewer):
    # 동일한 후보를 승인했을 때만 True를 반환하세요.
    pass

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def can_apply(expected, candidate, approved_by_reviewer):
    return expected == candidate and approved_by_reviewer

print(can_apply(approved, candidate, True))
print(can_apply(approved, approved, False))
print(can_apply(approved, approved, True))</code></pre>
<p>실행 결과</p><pre><code>False
False
True</code></pre>
<p>첫 호출에서는 승인 뒤 후보 내용이 달라졌고, 둘째에서는 후보가 같아도 승인이 없습니다. 셋째 결과만 적용 조건을 충족합니다. 적용 뒤 올바른 출처를 반환하는지는 다음 셀에서 따로 확인합니다.</p>
</details>


## 기능 검사 실패 시 복구 후보 선택

아래 배포 대상은 수업용 메모리 사전입니다. 검사된 후보를 적용한 뒤 과제 안내 질문이 근거 ID를 유지하는지 확인합니다. 실패하면 보관한 이전 이미지로 돌립니다. DB 내용은 이 작업에서 바뀌지 않습니다.


### health와 출처의 두 검사

<p>버전 B는 health 200을 반환하지만 source는 None입니다. 이 예제의 통과 기준에는 출처 course-guide-v1:p2도 있으므로 정상 HTTP 상태만 놓고 적용을 확정할 수 없습니다.</p>


### 셀 03 · 검증 실패 시 롤백 판단

health가 정상이더라도 문서 근거가 사라진 후보를 어떻게 처리하는지 보세요.

앞 과제에서는 승인이 가리키는 후보와 적용할 후보가 달라서 멈췄습니다. 이번에는 B의 승인이 유효해 적용했고, 그 뒤 검사가 실패한 상황입니다. 되돌리기를 판단하는 시점이 다릅니다.


In [3]:
versions = {"A": {"health": 200, "source": "course-guide-v1:p2"},
            "B": {"health": 200, "source": None}}
deployment = {"active": "A"}
previous = deployment["active"]
deployment["active"] = "B"
current = versions[deployment["active"]]
if current["health"] != 200 or current["source"] != "course-guide-v1:p2":
    deployment["active"] = previous
print(deployment["active"], versions[deployment["active"]]["source"])

A course-guide-v1:p2


A와 원래 근거 ID가 보입니다. 이 결과는 복구 조건을 로컬에서 재현한 것입니다. 실제 배포에서는 이전 이미지와 현재 스키마가 호환되는지, 이전 버전에서도 같은 문서가 조회되는지 별도 검증해야 합니다.


### 이전 버전 포인터 복구하기

<p>B를 적용하기 전에 A를 previous에 저장합니다. 기능 검사에 실패하면 active를 previous로 돌리므로 마지막 출력은 A와 A에 기록된 출처입니다. 이 셀은 버전 선택을 딕셔너리로 모의합니다.</p>


### 과제 04 · 잘못된 출처를 반환한 후보의 복구 검사

후보 B도 HTTP 상태 200을 기록했지만 질문에 딸린 출처가 다른 문서입니다. 상태와 출처를 함께 검사한 뒤 후보가 탈락하면 이전판을 선택하고, 선택한 판에 같은 검사를 한 번 더 적용하세요.


In [ ]:
release = {"A": {"health": 200, "source": "course-guide-v1:p2"},
           "B": {"health": 200, "source": "다른-문서:p1"}}
previous, active = "A", "B"
expected_source = "course-guide-v1:p2"
# release[active]를 검사하고, 실패하면 previous를 선택해 같은 검사를 다시 실행하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>release = {&quot;A&quot;: {&quot;health&quot;: 200, &quot;source&quot;: &quot;course-guide-v1:p2&quot;},
           &quot;B&quot;: {&quot;health&quot;: 200, &quot;source&quot;: &quot;다른-문서:p1&quot;}}
previous, active = &quot;A&quot;, &quot;B&quot;
expected_source = &quot;course-guide-v1:p2&quot;

def accepts(version):
    record = release[version]
    return record[&quot;health&quot;] == 200 and record[&quot;source&quot;] == expected_source

print(active, accepts(active))
if not accepts(active):
    active = previous
print(active, accepts(active))</code></pre>
<p>실행 결과</p><pre><code>B False
A True</code></pre>
<p>후보 B에는 출처 값이 들어 있어도 기대한 문서와 다릅니다. 값이 비어 있는지만 검사하면 놓칩니다. A를 다시 선택한 뒤 같은 기능 검사를 통과해야 이 모의 복구가 끝납니다. 이 실습은 메모리 안의 버전 기록을 사용합니다.</p>
</details>


## 돌아보기

같은 후보를 승인받았더라도 적용 뒤 출처가 비어 있다면 버전 B의 기능 검사는 실패입니다. 적용 전 확인하는 지문과 적용 후 확인하는 문서 출처를 각각 어느 셀에서 읽는지 짚어 보세요.
